# 04-01 Embedding + 向量数据库

**RAG 的基础设施**：Embedding 将文本转换为向量，向量数据库高效存储和检索这些向量。

**本节目标**：
- 理解 Sentence Embedding 的生成流程
- 实战 FAISS（本地高性能）和 ChromaDB（持久化）
- 掌握 ANN 近似最近邻搜索原理
- 了解相似度度量：余弦、L2、内积

---

In [ ]:
import numpy as np
import sys, os
sys.path.insert(0, "..")
from utils.data_generator import generate_ad_knowledge_base

# 加载 B站广告知识库
knowledge_docs = generate_ad_knowledge_base()
print(f"知识库文档数: {len(knowledge_docs)}")
for i, doc in enumerate(knowledge_docs):
    print(f"  [{i}] {doc[:60]}...")

## 1. 生成 Embedding

In [ ]:
def get_embeddings(texts: list[str]) -> np.ndarray:
    """获取文本 Embedding（优先使用 sentence-transformers，否则用 mock）"""
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
        embs = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        print(f"使用 SentenceTransformer，维度: {embs.shape[1]}")
        return embs.astype(np.float32)
    except ImportError:
        # Mock embedding：基于文本哈希确保一致性
        print("使用 Mock Embedding（install sentence-transformers 获得真实效果）")
        dim = 128
        embs = []
        for text in texts:
            rng = np.random.default_rng(hash(text) % (2**32))
            v = rng.standard_normal(dim).astype(np.float32)
            v /= np.linalg.norm(v)  # L2 normalize
            embs.append(v)
        return np.array(embs)

doc_embeddings = get_embeddings(knowledge_docs)
print(f"Embedding 矩阵: {doc_embeddings.shape}  (docs × dim)")
print(f"L2 范数（应≈1.0 since normalized）: {np.linalg.norm(doc_embeddings[0]):.4f}")

## 2. FAISS 精确搜索

In [ ]:
import time

def build_faiss_index(embeddings: np.ndarray, use_ivf: bool = False):
    """构建 FAISS 索引"""
    try:
        import faiss
        dim = embeddings.shape[1]
        
        if use_ivf and len(embeddings) >= 100:
            # IVF（分桶）：适合大规模数据的近似搜索
            nlist = min(50, len(embeddings) // 5)  # 分桶数
            quantizer = faiss.IndexFlatIP(dim)
            index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
            index.train(embeddings)
            index.nprobe = 5  # 搜索时查几个桶（精度 vs 速度的权衡）
        else:
            # Flat：精确搜索，适合小规模
            index = faiss.IndexFlatIP(dim)  # Inner Product = 余弦相似度（normalized）
        
        index.add(embeddings)
        return index, "faiss"
        
    except ImportError:
        # Fallback: NumPy 实现
        return embeddings, "numpy"

def search_index(index_or_matrix, query_emb: np.ndarray, k: int = 3):
    """搜索最相近的 k 个文档"""
    if isinstance(index_or_matrix, np.ndarray):
        # NumPy fallback
        sims = index_or_matrix @ query_emb
        top_k = np.argsort(sims)[::-1][:k]
        return [(int(i), float(sims[i])) for i in top_k]
    else:
        # FAISS
        q = query_emb.reshape(1, -1)
        scores, indices = index_or_matrix.search(q, k)
        return [(int(indices[0][i]), float(scores[0][i])) for i in range(k)]

index, backend = build_faiss_index(doc_embeddings)
print(f"索引构建完成（backend: {backend}）")

# 测试搜索
queries = [
    "CTR点击率怎么计算",
    "广告审核需要多久",
    "信息流广告收费方式",
]

query_embs = get_embeddings(queries)

for query, q_emb in zip(queries, query_embs):
    results = search_index(index, q_emb, k=2)
    print(f"\n问题: {query!r}")
    for idx, score in results:
        print(f"  [相似度={score:.3f}] {knowledge_docs[idx][:70]}...")

## 3. ChromaDB（持久化向量库）

In [ ]:
try:
    import chromadb
    
    # 内存模式（测试用）
    client = chromadb.Client()
    collection = client.create_collection(
        name="bilibili_ad_kb",
        metadata={"hnsw:space": "cosine"}  # 使用余弦相似度
    )
    
    # 添加文档
    collection.add(
        documents=knowledge_docs,
        ids=[f"doc_{i}" for i in range(len(knowledge_docs))],
        metadatas=[{"source": "bilibili_ad_guide", "doc_idx": i} 
                   for i in range(len(knowledge_docs))],
        embeddings=doc_embeddings.tolist(),
    )
    
    print(f"ChromaDB collection: {collection.count()} 个文档")
    
    # 搜索
    query_text = "如何提升广告CTR"
    q_emb = get_embeddings([query_text])[0]
    
    results = collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=3,
        include=["documents", "distances", "metadatas"]
    )
    
    print(f"\n查询: {query_text!r}")
    for doc, dist, meta in zip(
        results["documents"][0],
        results["distances"][0],
        results["metadatas"][0]
    ):
        print(f"  [距离={dist:.3f}] {doc[:60]}...")
        
except ImportError:
    print("ChromaDB 未安装，展示 API 示意")
    print("""
ChromaDB 核心 API：
  client = chromadb.PersistentClient(path="./chroma_db")  # 持久化到磁盘
  collection = client.get_or_create_collection("my_kb")
  
  # 添加
  collection.add(documents=[...], ids=[...], embeddings=[...])
  
  # 查询
  results = collection.query(query_embeddings=[...], n_results=3)
  
  # 过滤（元数据）
  results = collection.query(..., where={"source": "bilibili"})
    """)

## 4. 向量数据库对比

| 特性 | FAISS | ChromaDB | Pinecone | Qdrant |
|------|-------|----------|---------|--------|
| 部署 | 本地库 | 本地/服务 | 云服务 | 本地/云 |
| 持久化 | 手动序列化 | ✅ 自动 | ✅ 云端 | ✅ 自动 |
| 元数据过滤 | ❌ | ✅ | ✅ | ✅ |
| 规模 | 亿级 | 百万级 | 亿级 | 亿级 |
| 适用场景 | 研究/本地 | 学习/原型 | 生产/云端 | 生产/自托管 |

## 面试速记

| 问题 | 要点 |
|------|------|
| ANN vs 精确搜索 | ANN（HNSW/IVF）以精度换速度，亿级数据下 10ms 内完成检索 |
| 为什么用余弦相似度 | 不受向量长度影响，只看方向（语义方向）; L2 受文本长度影响 |
| FAISS IndexFlatIP vs L2 | IP（内积）= 余弦（已normalize）；L2 适合非 normalize 场景 |
| Embedding 维度选择 | 高维→表达力强但慢；常用 384/768/1536，RAG 用 384-768 足够 |

**下一节**: `02_chunking_strategies.ipynb`